---
subject: python-base, 10 pages
---

# Python: Functions and Classes

## 1. Functions
Two common categories of functions in Python are:
- Built-in functions, which are predefined and readily available.
- User-defined functions, which organize reusable logic into named code blocks.

### 1.1. Function as object

In [0]:
print.__name__

Now assign the alias `pr` to `print()` and observe its behavior.

In [0]:
pr = print
pr('Lion')

In [0]:
pr.__name__

### 1.2. User-defined functions
Define a function with the `def` statement. Arguments provide the function's inputs, and the `return` statement specifies its output. A function without an explicit `return` statement returns `None`.

:::{note}
- Aim for one clear responsibility per function; it makes code easier to test and reason about.
- Add a *docstring* that explains what the function does.
- Use type hints (`:` for parameters and `->` for the return value) to annotate expected types. Type hints are optional but are used by IDEs, linters, and static type checkers.
:::

In [0]:
def exp(base: float, power: int) -> float:
    """Compute base raised to power."""
    return base ** power

In [0]:
exp(2, 3)

In [0]:
exp(base=-5, power=2)

In [0]:
exp.__doc__

#### Default values

In [0]:
def exp(base, power=2):
    return base ** power
exp(base=5)

In [0]:
exp(base=3, power=4)

In [0]:
exp(3)

#### Variable scope
Names assigned inside a function are local by default. The `global` statement declares that a name refers to a variable in the module's global scope.

In [0]:
y = 100

def f(x):
    y = x + 7
    return y

print(f(10))
print(y)

In [0]:
y = 100

def f(x):
    global y
    y = x + 7
    return y

print(f(10))
print(y)

:::{admonition} Pitfall
:class: danger

Avoid `global` in production code; return values instead. Global state creates hidden dependencies that make code harder to understand, test, and maintain.

:::

#### Returning multiple values
A function can package several values into a tuple, which the caller can unpack into separate variables.

In [0]:
def rectangle(width, length):
    perimeter = 2 * (width + length)
    area = width * length
    return perimeter, area

# unpacking the output
perimeter, area = rectangle(10, 15)

#### Variable-length argument lists
Python provides `*args` and `**kwargs` for accepting variable-length argument lists. The names `args` and `kwargs` are conventional rather than required.
- `*args` collects additional positional arguments into a tuple.
- `**kwargs` collects additional keyword arguments into a dictionary.

In [0]:
def mean_args(*args):
    mean = sum(args) / len(args)
    return mean
mean_args(1, 3, 5, 7)

In [0]:
def mean_kwargs(**kwargs):
    mean = sum(kwargs.values()) / len(kwargs)
    return mean
mean_kwargs(a=1, b=3, c=5, d=7)

#### Lambda functions
Python supports anonymous functions through the `lambda` expression. A lambda contains a single expression and is useful for short functions passed directly to higher-order functions.

In [0]:
# give the lambda function a name
square = lambda x: x*x
square(5)

In [ ]:
numbers = [-2, 6, 1, 0]
sorted(numbers, key=lambda x: 1 / x)

### 1.3. Higher-order functions
Higher-order functions such as `map()`, `sorted()`, and `filter()` accept other functions as arguments.

#### Mapping

In [0]:
# map from each word to its length
cats = ['tiger', 'lion', 'panther', 'cheetah', 'puma', 'jaguar', 'leopard']
list(map(len, cats))

In [0]:
# map from each number to its square
numbers = [-5, -4, -3, -2, -1, 1, 2, 3, 4, 5]
list(map(lambda x: x**2, numbers))

#### Sorting

In [0]:
# sort by item's length
cats = ['tiger', 'lion', 'panther', 'cheetah', 'puma', 'jaguar', 'leopard']
sorted(cats, key=len)

In [0]:
# sort by the reciprocal of each number
numbers = [-5, -4, -3, -2, -1, 1, 2, 3, 4, 5]
sorted(numbers, key=lambda x: 1/x)

#### Filtering

In [0]:
# filter out words containing 'e'
cats = ['tiger', 'lion', 'panther', 'cheetah', 'puma', 'jaguar', 'leopard']
list(filter(lambda x: 'e' in x, cats))

In [0]:
# filter out even numbers
numbers = [-5, -4, -3, -2, -1, 1, 2, 3, 4, 5]
list(filter(lambda x: x%2==0, numbers))

### 1.4. Decorator functions
A *decorator/wrapper* is a higher-order function that wraps an *inner function* with added functionalities. Decorators can feel abstract at first, so let’s start with how to use them before writing our own.

#### Practical usage
Let's say we want to apply the ReLU function $f(x)=\max(x,0)$ to a list of numbers. We break this problem into two steps, (1) write the `relu()` function that processes a single number and (2) applies it to the entire list. The second step will be implemented using Numpy's [`vectorize()`] function, as it is more readable than loops.

In this program, `vectorize` is the decorator function, and `relu()` is the inner function. It is written in two equivalent ways; notice that the second way provides a convenient syntax using the `@` symbol.

[`vectorize()`]: https://numpy.org/doc/stable/reference/generated/numpy.vectorize.html

In [0]:
from numpy import vectorize
x = range(-4, 5)

In [0]:
def relu(x):
    return x if x > 0 else 0
relu = vectorize(relu)

relu(x)

In [0]:
@vectorize
def relu(x):
    return x if x > 0 else 0

relu(x)

:::{note}
`vectorize()` is mainly for convenience/readability, not speed.
:::

#### Custom decorators
In this section, we will write some useful decorators from scratch. A convenience function called `wraps()` is utilized to preserve the original attributes of the inner function without altering the logic.

In [0]:
import logging
import datetime as dt
import functools

def timer(inner):
    @functools.wraps(inner)
    def _wrapper(*args, **kwargs):
        start = dt.datetime.now()
        
        # call the inner function
        output = inner(*args, **kwargs)
        
        end = dt.datetime.now()
        logging.warning(f'Elapsed time {end-start}')
        return output
    
    return _wrapper

In [0]:
import time

@timer
def exp(base, power=2):
    'The exponential function'
    time.sleep(0.5)
    return base**power

exp(3)

In [0]:
exp.__doc__

In [0]:
import time
def retry(n_retry):
    def retry_decorator(func):
        def _wrapper(*args, **kwargs):
            for _ in range(n_retry):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    time.sleep(0.1)
                    print(e)
        return _wrapper
    return retry_decorator

In [0]:
@retry(3)
def might_fail():
    print('Failed')
    return 1/0

In [0]:
might_fail()

### 1.5 Asynchronous I/O

Python’s `asyncio` lets you write coroutines (using `async def`) that can pause at `await` and resume later. You typically create several coroutines with `asyncio.gather()` and run them concurrently with `asyncio.run()`.

Think of Magnus Carlsen (a chess grandmaster) facing many amateurs. Instead of finishing one full game before starting the next (*synchronous*), he can make one move per board and keep moving on (*asynchronous*). Total time drops because the waiting time overlaps.

In [0]:
import asyncio
import time

ACCEL = 10
CARLSEN_MOVE = 5 / ACCEL
AMATEUR_MOVE = 20 / ACCEL
N_BOARDS = 3

start = time.perf_counter()
print(f'{time.perf_counter() - start:.1f} seconds: Start playing {N_BOARDS} boards asynchronously')

async def play_first_move(idx):
    time.sleep(CARLSEN_MOVE)
    print(f'{time.perf_counter() - start:.1f} seconds: Carlsen played first move on board {idx}')

    await asyncio.sleep(AMATEUR_MOVE)
    print(f'{time.perf_counter() - start:.1f} seconds: Opponent played first move on board {idx}')

async def main():
    tasks = [play_first_move(idx+1) for idx in range(N_BOARDS)]
    await asyncio.gather(*tasks)
    print(f'Total time: {time.perf_counter() - start:.1f} seconds')

await main()

## 2. Classes
Python is an [object-oriented programming] (OOP) language, meaning everything in Python is an *object*. Meanwhile, objects are *instances* of *classes*, the blueprints for creating objects. In this section, we learn how to invent our own classes to move to the next step of code reproducibility

[object-oriented programming]: https://en.wikipedia.org/wiki/Object-oriented_programming

### 2.1. Initialization
Use the statement `class` to inform creation of a class, followed by class name (use capitalized words according to [PEP 8] convention). The next step is defining the constructor via the `__init__()` special method. The first argument of this method is always the `self` keyword, representing the *instance* itself.

[PEP 8]: https://peps.python.org/pep-0008/

In [0]:
class Rectangle:
    'This class only stores sizes'
    def __init__(self, width, length):
        pass

In [0]:
Rectangle(3, 4).__doc__

#### Attributes
We start by adding some attributes to our class, which are variables defined inside classes. There are three types of attributes commonly seen in practice:
- *Normal attributes*, such as `width_` and `length`. They can be easily manipulated (accessed, modified, and deleted). Some libraries, such as Scikit-learn, name attributes with a trailing underscore to distinguish them from methods.
- *Private attributes* with a double leading underscore, like `__n_edges`, indicate *strong* internal use. These attributes invoke *name mangling*, which protects them from being modified. In practice, it is recommended by PEP 8 to write private attributes using a leading underscore like `_n_corners` (*weak* internal use). These work exactly the same as normal attributes, but people understand the convention and take extra care when dealing with them.
- *Magical attributes* with double leading and trailing underscores, like `__doc__` and `__name__`, are special ones that Python creates for you. You are free to override those to change the behavior of your class but should never make up such names.

In [0]:
class Rectangle:
    _n_corners = 4
    __n_edges = 4
    
    def __init__(self, width, length):
        self.width_ = width
        self.length = length
        
rec = Rectangle(3, 8)

In [0]:
rec.width_

In [0]:
rec._n_corners

In [0]:
# name mangling
rec._Rectangle__n_edges

In [0]:
import datetime as dt

In [0]:
now = dt.datetime.now()

In [0]:
now

In [0]:
print(now)

In [0]:
now.__str__()

In [0]:
now.__repr__()

#### Methods
Methods are functions defined inside of classes, with the first argument also being `self`. This allows methods to access attributes assigned during initialization. Additionally, there are some special methods that we can override:
- `__call__()` makes instances callable.
- `__str__()` alters the string to be printed of instances.
- `__repr__()` (abbreviated for *representation*) controls how instances are displayed. Specifically for IPython, it provides some overloading methods for [rich representation] of objects.

[rich representation]: https://ipython.readthedocs.io/en/stable/config/integrating.html#custom-methods

In [0]:
class Rectangle:
    def __init__(self, width, length):
        self.width = width
        self.length = length
        
    def compute_area(self):
        return self.width * self.length
    
    def compute_perimeter(self):
        return 2 * (self.width + self.length)
    
    def __call__(self):
        print(f'A rectangle size {self.width} x {self.length}')
        
    def __repr__(self):
        return f'Rec({self.width}x{self.length})'
    
    def __str__(self):
        return 'This is a rectangle'
        
rec = Rectangle(4, 6)

In [0]:
print(rec)

In [0]:
rec

In [0]:
rec.compute_area()

In [0]:
rec.compute_perimeter()

In [0]:
rec()

### 2.2. Special decorators

#### Class method
Class methods are methods that you can use on the class itself rather than on an instance. For example, you can write something like `Class.class_method()`, whereas the normal behavior would be `instance.normal_method()`. To define a class method, use the `@classmethod` decorator to wrap around a method, which always takes `cls` (an abbreviation for "class") as the first argument.

So when a standalone class is useful? Well, a popular use case is to provide addition ways to initialize instances. For example, instead of following exactly the blueprint of a rectangle by specifying width and length, we give our class more flexibility by allowing it to accept and process a string containing sizes. We can see this kind of method appears in Pandas's
[`DataFrame.from_dict()`] method.

[`DataFrame.from_dict()`]: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.from_dict.html

In [0]:
class Rectangle:
    def __init__(self, width, length):
        self.width = width
        self.length = length
    
    @classmethod
    def from_string(cls, string_args):
        width, length = [float(edge) for edge in string_args.split(',')]
        return cls(width, length)
    
    @classmethod
    def from_total_diff(cls, total, diff):
        width = (total - diff) / 2
        length = (total + diff) / 2
        return cls(width, length)
    
    def compute_area(self):
        return self.width * self.length
    
    def compute_perimeter(self):
        return 2 * (self.width + self.length)

In [0]:
Rectangle.from_string('2.5, 4').compute_perimeter()

In [0]:
Rectangle.from_total_diff(8, 2).compute_area()

#### Static method
A static method, like a class method, is defined in a class, but does not take `self` or `cls` as its first argument. Static methods behave like normal (standalone) functions, but are grouped inside a class because they have a logical relationship to it. Use them when a method doesn’t need to access any instance or class data. You can create one with the decorator `@staticmethod`, but they are rarely seen in practice.

#### Property
A property is a special attribute that gives developers maximum access over three basic methods: *getter*, *setter* and *deleter*. There are two use cases that involve properties being (1) logging, which is useful for setter as well as deleter and (2) adding constraints to the setter method. The second use case enables *data validation*, an ordinary task in Data Science projects. In this section, we are going to create a property *radius* for the class *Circle* using several syntaxes of the [`@property`] decorator.
- First syntax: We define three methods separately: getter, setter, and deleter. Then, we pass all three to the `property()` function, which returns a `property` object. This object becomes an attribute for instances of this class.
- Second syntax: Instead of passing all three methods to the constructor, we utilize the `@property` decorator along with `setter()` and `deleter()` to add each corresponding method one-by-one. The advantage of this syntax is that we don't need to worry about inventing new method names, thereby improving code readability.

[`@property`]: https://docs.python.org/3/library/functions.html#property

In [0]:
import logging
import numpy as np

In [0]:
class Circle:
    def __init__(self, radius=0):
        self._radius = radius
    
    def _get_radius(self):
        return self._radius
    
    def _set_radius(self, value):
        if value < 0:
            logging.error('Radius should be positive')
        return self
    
    def _del_radius(self):
        logging.warning('Attribute radius deleted')
        del self._radius
        return self
    
    radius = property(_get_radius, _set_radius, _del_radius)
    
    def compute_area(self):
        return np.pi * self.radius**2

In [0]:
class Circle:
    def __init__(self, radius=0):
        self._radius = radius
    
    @property
    def radius(self):
        return self._radius
    
    @radius.setter
    def radius(self, value):
        if value < 0:
            logging.error('Radius should be positive')
        return self
    
    @radius.deleter
    def radius(self):
        logging.warning('Attribute radius deleted')
        del self._radius
        return self
    
    def compute_area(self):
        return np.pi * self.radius**2

In [0]:
circle = Circle()
circle.radius

In [0]:
del circle.radius

In [0]:
circle.radius = -1

In [0]:
circle.radius = 10
circle.compute_area()

### 2.3. The four pillars
In this section we will go over four pillars of OOP and examples in Python.

#### Abstraction
Abstraction is the concept of hiding the real implementation and showing only necessary information of an application. For example, when solving a quadratic equation, we do not care about the discriminant ($\Delta$) as well as complicated conditions and formulas, only need to know what the solutions are. In the below script, abstraction is represented via *internal* methods and attributes, which can be recognized with a leading underscore.

In [0]:
import numpy as np

class Quadratic:
    def __init__(self, a, b, c):
        self._a = a
        self._b = b
        self._c = c
        self._discriminant = b**2 - 4*a*c
    
    def _repr_latex_(self):
        return f'${self._a}x^2 + {self._b}x + {self._c}$'
    
    def _solution_formula_1(self, a, b, discriminant):
        return (-b - np.sqrt(discriminant)) / (2*a)
    
    def _solution_formula_2(self, a, b, discriminant):
        return (-b + np.sqrt(discriminant)) / (2*a)
    
    def solve(self):
        if self._discriminant < 0:
            return False
        if self._discriminant == 0:
            sol = self._solution_formula_1(self._a, self._b, self._discriminant)
            return sol
        if self._discriminant > 0:
            sol1 = self._solution_formula_1(self._a, self._b, self._discriminant)
            sol2 = self._solution_formula_2(self._a, self._b, self._discriminant)
            return sol1, sol2

In [0]:
eq = Quadratic(1,-4,3)
eq

In [0]:
eq.solve()

#### Encapsulation
Encapsulation is the manipulation of class privacy. I.e., you can remove access completely for specific attributes (by making them private), or limit others to be read-only (by designing properties with only the getter method). For example, a triangle always has 3 edges, this is a constant and should be read-only. It will be implemented as a property with the getter method only.

In [0]:
class Triangle:
    @property
    def n_edges(self):
        return 3
    
    def __init__(self, *args):
        if len(args) != self.n_edges:
            raise ValueError(f'A triangle should have exactly {self.n_edges} edges. Got {len(args)}.')
        self.edges = args
    
    def compute_perimeter(self):
        return sum(self.edges)

triangle = Triangle(3,4,5)
triangle.n_edges = 4

#### Inheritance
Inheritance refers to the situation where a child class inherits all methods and attributes of a parent class. This principle is applied when a class represents a specialized case of a more general one, defined in Python with the syntax `Child(Parent)`. For instance, a square is a specialized case of a rectangle, where the width and length are equal. Therefore, we define the Square class to inherit everything from the Rectangle class, with slight modifications during initialization. The `super()` function is utilized to invoke the parent class, enabling flexible modification of its behavior.

There are also more complex strategies: *multiple inheritance* with the syntax `Child(Father, Mother)` and *hierarchical inheritance* with the syntax `Child(Father(Grandpa))`. However, these are advanced topics and will not be discussed in this section.

In [0]:
class Rectangle:
    def __init__(self, width, length):
        self.width = width
        self.length = length
        
    def compute_area(self):
        return self.width * self.length
    
    def compute_perimeter(self):
        return 2 * (self.width + self.length)

class Square(Rectangle):
    def __init__(self, size):
        super().__init__(size, size)

Square(5).compute_perimeter()

#### Polymorphism
Polymorphism is the concept that an operator or a function can accept different types of input. For example:
- the `+` operator can be used to concatenate strings or add numbers.
- the `len()` function can be used to determine the size of lists or dictionaries.

In [0]:
1 + 4

In [0]:
'py' + 'thon'

In [0]:
len([1,2,3,4,5])

In [0]:
len(dict(a=1, b=2, c=3))

## Resources
- bas.codes - [Understanding decorators in Python](https://bas.codes/posts/python-decorators)
- towardsdatascience.com - [Python: Decorators in OOP](https://towardsdatascience.com/python-decorators-in-oop-3189c526ead6)